<a href="https://colab.research.google.com/github/umar-zama/aws-iam-privesc-scanner/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/umar-zama/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/umar-zama/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "duckdb", "huggingface_hub"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")
print("Working dir:", os.getcwd())

Working dir: /content/flyrank-ml-internship


In [ ]:
import duckdb, pandas as pd, numpy as np
from google.colab import userdata
from huggingface_hub import hf_hub_download

hf_token = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.sql("INSTALL httpfs;")
con.sql("LOAD httpfs;")
con.sql(f"CREATE OR REPLACE SECRET hf_secret (TYPE huggingface, TOKEN '{hf_token}');")

local_path = hf_hub_download(repo_id="FlyRank/internship-warehouse", repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet", token=hf_token)
dim_content_path = hf_hub_download(repo_id="FlyRank/internship-warehouse", repo_type="dataset",
    filename="dim_content.parquet", token=hf_token)

pages = con.sql(f"""
    SELECT
        f.content_hash_id, f.client_hash_id, f.report_date,
        c.content_type, c.search_volume, c.competition, c.competition_level, c.cpc,
        c.main_intent, c.backlinks, c.category_count, c.word_count, c.char_count,
        c.last_optimized_date,
        CASE WHEN f.gsc_avg_position <= 3 THEN 'top_3'
             WHEN f.gsc_avg_position <= 10 THEN 'page_1'
             WHEN f.gsc_avg_position <= 20 THEN 'page_2'
             ELSE 'deep' END AS position_tier,
        f.gsc_impressions, f.gsc_clicks,
        CASE WHEN f.gsc_impressions > 0 THEN f.gsc_clicks * 1.0 / f.gsc_impressions ELSE NULL END AS ctr
    FROM read_parquet('{local_path}') f
    JOIN read_parquet('{dim_content_path}') c ON f.content_hash_id = c.content_hash_id
    WHERE f.gsc_impressions >= 50
""").df()

pages["report_date"] = pd.to_datetime(pages["report_date"])
pages["last_optimized_date"] = pd.to_datetime(pages["last_optimized_date"])
pages["days_since_optimized"] = (pages["report_date"] - pages["last_optimized_date"]).dt.days
pages = pages.dropna(subset=["ctr"])
print(f"Pages loaded: {len(pages):,}")

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

dim_content.parquet: reconstructing file:   0%|          |  0.00B / 19.6MB            

dim_content.parquet: downloading bytes:           |  0.00B            

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Pages loaded: 1,037,442


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Method: Random Forest Regressor, predicting ctr directly.

Why: my Week-4 baseline only explains ctr using a group average over (position_tier, content_type). A tree-based model can use richer content signals (search_volume, competition, cpc, main_intent, backlinks, word_count, days_since_optimized) without me hand-picking interactions, and handles mixed numeric/categorical features without heavy preprocessing.

clicks is never used as a feature — it defines ctr, so including it would be circular (the w03 leakage trap).

In [ ]:
print("Rows:", len(pages))
print("Columns available:", list(pages.columns))

Rows: 1037442
Columns available: ['content_hash_id', 'client_hash_id', 'report_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'word_count', 'char_count', 'last_optimized_date', 'position_tier', 'gsc_impressions', 'gsc_clicks', 'ctr', 'days_since_optimized']


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Split: client-holdout (grouped by client_hash_id), not a random row split. Whole clients are held out for testing so the model is validated on clients it never trained on — pages from the same client can share patterns a random split would let the model memorize.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(splitter.split(pages, groups=pages["client_hash_id"]))
train_df, test_df = pages.iloc[train_idx].copy(), pages.iloc[test_idx].copy()

print(f"Train: {len(train_df):,} rows, {train_df['client_hash_id'].nunique()} clients")
print(f"Test:  {len(test_df):,} rows, {test_df['client_hash_id'].nunique()} clients")
overlap = set(train_df["client_hash_id"]) & set(test_df["client_hash_id"])
print(f"Client overlap: {len(overlap)} (should be 0)")

Train: 646,002 rows, 28 clients
Test:  391,440 rows, 13 clients
Client overlap: 0 (should be 0)


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import mean_absolute_error, r2_score

cat_cols = ["position_tier", "content_type", "main_intent", "competition_level"]
num_cols = ["search_volume", "competition", "cpc", "backlinks", "category_count",
            "word_count", "char_count", "days_since_optimized", "gsc_impressions"]

encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
train_cat = encoder.fit_transform(train_df[cat_cols].fillna("missing"))
test_cat = encoder.transform(test_df[cat_cols].fillna("missing"))

X_train = np.hstack([train_cat, train_df[num_cols].fillna(0).values])
X_test = np.hstack([test_cat, test_df[num_cols].fillna(0).values])
y_train, y_test = train_df["ctr"], test_df["ctr"]

model = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
model_preds = model.predict(X_test)

train_group_means = train_df.groupby(["position_tier", "content_type"])["ctr"].mean()
baseline_preds = test_df.set_index(["position_tier", "content_type"]).index.map(train_group_means)
baseline_preds = pd.Series(baseline_preds, index=test_df.index).fillna(train_df["ctr"].mean())

comparison = pd.DataFrame({
    "method": ["Baseline (group mean)", "Random Forest"],
    "MAE": [mean_absolute_error(y_test, baseline_preds), mean_absolute_error(y_test, model_preds)],
    "R2": [r2_score(y_test, baseline_preds), r2_score(y_test, model_preds)],
})
comparison

,method,MAE,R2
0,Baseline (group mean),0.004361,0.003573
1,Random Forest,0.004233,0.001529


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(model, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1)
importance_df = pd.DataFrame({
    "feature": cat_cols + num_cols, "importance": perm.importances_mean
}).sort_values("importance", ascending=False)
print(importance_df)

test_df["model_pred"] = model_preds
test_df["abs_error"] = (test_df["ctr"] - test_df["model_pred"]).abs()
print("\nWorst 5 predictions:")
print(test_df.sort_values("abs_error", ascending=False)[["content_type","position_tier","ctr","model_pred","abs_error"]].head())

                 feature    importance
9             word_count  1.352828e-01
10            char_count  8.283890e-02
11  days_since_optimized  3.206978e-02
4          search_volume  3.029918e-02
8         category_count  2.246318e-02
7              backlinks  1.428331e-02
0          position_tier  1.276664e-02
12       gsc_impressions  2.440285e-03
5            competition  2.615682e-04
2            main_intent  7.005386e-05
1           content_type -1.898524e-07
3      competition_level -7.403891e-06
6                    cpc -1.883864e-05

Worst 5 predictions:
           content_type position_tier       ctr  model_pred  abs_error
194226   feedly article         top_3  0.133333    0.003255   0.130078
872810   feedly article         top_3  0.115385    0.003264   0.112121
956902  keyword article        page_1  0.109375    0.002647   0.106728
153237  keyword article        page_1  0.100000    0.003709   0.096291
357965  keyword article         top_3  0.098361    0.002119   0.096242


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.